# 03 — Analyse exploratoire de Hannousse & Yahiouche 2021 (Mendeley)

**Objectif** : vérifier que ce jeu (prévu comme jeu principal d'entraînement) ne contient pas le raccourci de PhiUSIIL, puis chercher ses propres défauts : doublons, codes « inconnu », caractéristiques biaisées.

**Données** : `data/raw/hannousse_2021/dataset_B_05_2020.csv` — 11 430 URL, 87 caractéristiques (56 URL, 24 contenu, 7 externes), étiquette `status` en texte.

In [1]:
import pandas as pd

## 1. Taille et équilibre des classes

In [3]:
df = pd.read_csv("../data/raw/hannousse_2021/dataset_B_05_2020.csv")
print(df.shape)
print(df["status"].value_counts())

(11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


**Résultat** : 11 430 URL, **5 715 / 5 715** (50/50), 89 colonnes = 87 caractéristiques + `url` + `status`.

## 2. Forme des URL légitimes (comparaison avec PhiUSIIL)

In [4]:
df[["url", "status"]].sample(10, random_state=0)


,url,status
6406,https://www.disabilitysecrets.com/,legitimate
4266,http://mnsglobal.ge/a2/login/mobile.de/tanVali...,phishing
1238,http://idealcaisse.fr/cs/,phishing
2646,http://xmley.codesandbox.io/index3.html,phishing
2217,http://www.amsoil.com/shop/by-product/transmis...,legitimate
2158,http://vps-2303d288.vps.ovh.net/,phishing
9823,http://tvline.com/2016/10/17/switched-at-birth...,legitimate
4401,http://www.klik.am/EEUVB/,phishing
9349,http://excel.bigresource.com/Mining-Text-Field...,legitimate
1399,http://wiki.secondlife.com/wiki/Second_Life_Ra...,legitimate


**Résultat** : légitimes **variées** — pages internes (`/shop/…`, `/wiki/…`), souvent en `http://`, avec ou sans `www.`. Phishing réaliste (hébergeurs gratuits, VPS, sites compromis).

## 3. La regex de PhiUSIIL appliquée à Hannousse

**Exactement** la même règle qu'en `02`, sans l'adapter : la question est « le raccourci de PhiUSIIL fonctionne-t-il sur un autre jeu ? ».

In [5]:
pattern = r"https://www\.[^/?#]+"
df["legit_shape"] = df["url"].str.fullmatch(pattern)
df.groupby("status")["legit_shape"].mean()


status
legitimate    0.005074
phishing      0.000875
Name: legit_shape, dtype: float64

In [6]:
from sklearn.metrics import confusion_matrix, classification_report

In [7]:


y_true = df["status"]
y_pred = df["legit_shape"].map({True: "legitimate", False: "phishing"})

labels = ["phishing", "legitimate"]
print(confusion_matrix(y_true, y_pred, labels=labels))
print(classification_report(y_true, y_pred, labels=labels, digits=4))


[[5710    5]
 [5686   29]]
              precision    recall  f1-score   support

    phishing     0.5011    0.9991    0.6674      5715
  legitimate     0.8529    0.0051    0.0101      5715

    accuracy                         0.5021     11430
   macro avg     0.6770    0.5021    0.3387     11430
weighted avg     0.6770    0.5021    0.3387     11430



**Résultat** : la regex ne reconnaît que 0,5 % des légitimes → précision 0,5011, F1 0,6674, accuracy **0,5021** (hasard). **Le raccourci de PhiUSIIL n'existe pas ici.**

## 4. Doublons d'URL et de caractéristiques

In [8]:
print("URL en double :", df["url"].duplicated().sum())
print("Lignes entièrement identiques :", df.duplicated().sum())


URL en double : 1
Lignes entièrement identiques : 0


In [9]:
# 1. Les deux lignes de l'URL en double
display(df[df["url"].duplicated(keep=False)][["url", "status"]])

# 2. Doublons sur les caractéristiques seules (sans l'URL)
features = df.drop(columns=["url", "status"])
print("Caractéristiques identiques :", features.duplicated().sum())

# 3. Parmi eux, combien avec des étiquettes différentes ?
dup = df[features.duplicated(keep=False)]
conflicts = dup.groupby(list(features.columns))["status"].nunique()
print("Groupes en conflit :", (conflicts > 1).sum())


,url,status
6035,http://e710z0ear.du.r.appspot.com/c:/users/use...,phishing
11305,http://e710z0ear.du.r.appspot.com/c:/users/use...,phishing


Caractéristiques identiques : 174
Groupes en conflit : 0


**Résultat** : 1 URL en double (même URL collectée à 13 jours d'écart) ; **174 doublons de caractéristiques**, tous du phishing, **0 conflit** d'étiquette.

## 5. Les doublons de caractéristiques partagent-ils le même domaine ?

Si oui, le découpage par domaine (`04_split`) les gardera du même côté. Sinon, ce sont des campagnes multi-domaines qui peuvent fuiter entre train et test.

In [10]:
import tldextract

In [11]:
df["domain"] = df["url"].apply(lambda u: tldextract.extract(u).registered_domain)

features = df.drop(columns=["url", "status", "domain"])
dup = df[features.duplicated(keep=False)].copy()
dup["group"] = dup.groupby(list(features.columns)).ngroup()

per_group = dup.groupby("group")["domain"].nunique()
print("Groupes avec un seul domaine :", (per_group == 1).sum())
print("Groupes avec plusieurs domaines :", (per_group > 1).sum())


Groupes avec un seul domaine : 114
Groupes avec plusieurs domaines : 3


/tmp/ipykernel_84046/3013757407.py:1: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  df["domain"] = df["url"].apply(lambda u: tldextract.extract(u).registered_domain)


In [12]:
multi = per_group[per_group > 1].index
display(dup[dup["group"].isin(multi)].sort_values("group")[["group", "url", "domain", "status"]])


,group,url,domain,status
182,4,http://m.hf695.com/,hf695.com,phishing
10398,4,http://m.hf215.com/,hf215.com,phishing
4178,5,http://m.hf735.com/,hf735.com,phishing
2331,5,http://m.hf356.com/,hf356.com,phishing
787,6,http://m.g2234.com/,g2234.com,phishing
5769,6,http://m.g2236.com/,g2236.com,phishing


**Résultat** : 114 groupes de doublons sur 117 tiennent dans un seul domaine ; 3 sont des **campagnes multi-domaines** (`hf695.com` / `hf215.com`, `hf735.com` / `hf356.com`, `g2234.com` / `g2236.com`) → **décision : retirer les doublons de caractéristiques avant le découpage**.

## 6. Valeurs manquantes et codes « inconnu »

In [14]:
print("Valeurs manquantes :", df.isna().sum().sum())
print(df.dtypes.value_counts())



Valeurs manquantes : 0
int64      74
float64    13
str         3
bool        1
Name: count, dtype: int64


In [15]:
# Valeurs négatives = codes « inconnu » (-1, -2) dans les scripts des auteurs
neg = (df.select_dtypes("number") < 0).sum()
print(neg[neg > 0])

domain_registration_length      46
domain_age                    1837
dtype: int64


**Résultat** : 0 valeur manquante, 87 colonnes numériques. Codes « inconnu » cachés : `domain_age` (-1, -2, -12 ; 1 837 lignes, 16 %) et `domain_registration_length` (-1 ; 46 lignes) → **décision : `NaN` puis médiane du train + indicatrice, dans le `Pipeline`**.

## 7. Caractéristiques externes : que signifie un `0` ?

D'après `external_features.py` des auteurs : pour `google_index`, `dns_record`, `whois_registered_domain`, **`0` = bon signe** (indexé, résolu, cohérent) ; pour `web_traffic` et `page_rank`, `0` = pas de score **ou** requête échouée.

In [16]:
external = ["web_traffic", "page_rank", "google_index", "dns_record",
            "domain_age", "domain_registration_length", "whois_registered_domain"]

(df[external] == 0).groupby(df["status"]).mean().T.round(3)


status,legitimate,phishing
web_traffic,0.146,0.632
page_rank,0.018,0.449
google_index,0.831,0.101
dns_record,0.997,0.963
domain_age,0.000,0.001
domain_registration_length,0.113,0.133
whois_registered_domain,0.945,0.910


**Résultat** : `google_index` (83 % des légitimes indexées contre 10 % du phishing), `page_rank` et `web_traffic` séparent fortement les classes : ils mesurent la **notoriété** du site, pas le phishing.

## 8. Ligne de base « popularité » : `google_index == 1` → phishing

Une seule colonne, aucun ML : jusqu'où la seule notoriété du site permet-elle de classer ?

In [17]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_google = df["google_index"].map({1: "phishing", 0: "legitimate"})

labels = ["phishing", "legitimate"]
print(confusion_matrix(df["status"], y_pred_google, labels=labels))
print(classification_report(df["status"], y_pred_google, labels=labels, digits=4))


[[5136  579]
 [ 967 4748]]
              precision    recall  f1-score   support

    phishing     0.8416    0.8987    0.8692      5715
  legitimate     0.8913    0.8308    0.8600      5715

    accuracy                         0.8647     11430
   macro avg     0.8664    0.8647    0.8646     11430
weighted avg     0.8664    0.8647    0.8646     11430



## Conclusions : premier aperçu de Hannousse 2021

**Données** : 11 430 URL, 87 caractéristiques, 5 715 phishing / 5 715 légitimes, 0 valeur manquante.

**Le raccourci de PhiUSIIL n'existe pas ici** : les légitimes incluent des pages internes, en `http://`, sans `www.`.

| Règle (sans ML) | Dataset | Précision | Rappel | F1 | Accuracy |
| --- | --- | --- | --- | --- | --- |
| Regex `https://www.domaine` | PhiUSIIL | 1,0000 | 0,9938 | 0,9969 | 0,9973 |
| Regex `https://www.domaine` | Hannousse | 0,5011 | 0,9991 | 0,6674 | 0,5021 |
| `google_index == 1` | Hannousse | 0,8416 | 0,8987 | 0,8692 | 0,8647 |

**Problèmes trouvés :**
- 1 URL en double (collectée 2 fois à 13 jours d'écart, `length_url` différent).
- 174 doublons de caractéristiques, tous phishing, 0 conflit ; 3 campagnes multi-domaines (`hf*.com`, `g22*.com`).
- Codes « inconnu » : `domain_age` (-1, -2, -12 ; 16 %), `domain_registration_length` (-1).
- Caractéristiques externes (`google_index`, `page_rank`, `web_traffic`) = popularité du site → une seule colonne donne 86,5 %.

**Décisions :**
- Retirer les doublons de caractéristiques avant le découpage.
- Indicatrice « inconnu » + médiane (calculée sur le train) pour `domain_age` et `domain_registration_length`.
- Résultats présentés avec et sans caractéristiques externes ; plancher à battre = 86,5 %.
